# XGBoost Seed Ensemble

`09_submission_xgboost_seed_search.csv` 的公榜分数为 `0.12485`，说明 XGBoost 随机种子方向有效。本 notebook 在 `09` 的基础上继续优化：不改变数据处理和模型参数，只将多个表现好的 XGBoost seeds 做平均，再与 Lasso 融合。

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train.shape, test.shape

((1460, 81), (1459, 80))

## 1. 复用当前最佳数据处理

In [2]:
target = "SalePrice"
id_column = "Id"

outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
train_model = train.loc[~outlier_mask].copy()

X = train_model.drop(columns=[target, id_column])
y = train_model[target]
X_test = test.drop(columns=[id_column]).copy()
test_ids = test[id_column]

print("移除异常点数量:", int(outlier_mask.sum()))
print("异常点 Id:", train.loc[outlier_mask, id_column].tolist())

移除异常点数量: 2
异常点 Id: [524, 1299]


In [3]:
def log_transform_skewed_features(X_train: pd.DataFrame, X_test: pd.DataFrame, threshold: float = 0.75):
    X_train = X_train.copy()
    X_test = X_test.copy()
    numeric_columns = X_train.select_dtypes(include="number").columns
    skewness = X_train[numeric_columns].skew().sort_values(ascending=False)
    skewed_columns = skewness[skewness > threshold].index.tolist()

    transformed_columns = []
    for column in skewed_columns:
        min_value = min(X_train[column].min(), X_test[column].min())
        if pd.notna(min_value) and min_value >= 0:
            X_train[column] = np.log1p(X_train[column])
            X_test[column] = np.log1p(X_test[column])
            transformed_columns.append(column)

    return X_train, X_test, transformed_columns


X_log, X_test_log, transformed_columns = log_transform_skewed_features(X, X_test)
print("log1p 处理的列数量:", len(transformed_columns))

log1p 处理的列数量: 20


## 2. 建模函数

In [4]:
numeric_features = X_log.select_dtypes(include="number").columns.tolist()
categorical_features = X_log.select_dtypes(exclude="number").columns.tolist()


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_model(base_model, scale_numeric: bool = False) -> TransformedTargetRegressor:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(numeric_steps), numeric_features),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_one_hot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    return TransformedTargetRegressor(
        regressor=Pipeline(steps=[("preprocessor", preprocessor), ("model", base_model)]),
        func=np.log1p,
        inverse_func=np.expm1,
    )


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return float(np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))))

## 3. OOF 验证 Seed Ensemble

搜索结果显示 `seed=23`、`seed=73` 和 `seed=9` 的平均最优，融合权重为 `0.60 * Lasso + 0.40 * XGBoost均值`。

In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

lasso_model = build_model(
    LassoCV(alphas=np.logspace(-4, 0, 40), cv=5, max_iter=30000, random_state=42),
    scale_numeric=True,
)

xgb_params = dict(
    n_estimators=1100,
    learning_rate=0.025,
    max_depth=3,
    subsample=0.80,
    colsample_bytree=0.70,
    reg_lambda=7,
    reg_alpha=0.01,
    objective="reg:squarederror",
    n_jobs=-1,
)
selected_seeds = [23, 73, 9]

lasso_oof = np.zeros(len(X_log))
xgb_oofs = {seed: np.zeros(len(X_log)) for seed in selected_seeds}

for train_idx, valid_idx in cv.split(X_log):
    fold_lasso = clone(lasso_model)
    fold_lasso.fit(X_log.iloc[train_idx], y.iloc[train_idx])
    lasso_oof[valid_idx] = fold_lasso.predict(X_log.iloc[valid_idx])

    for seed in selected_seeds:
        fold_xgb = build_model(
            XGBRegressor(random_state=seed, **xgb_params),
            scale_numeric=False,
        )
        fold_xgb.fit(X_log.iloc[train_idx], y.iloc[train_idx])
        xgb_oofs[seed][valid_idx] = fold_xgb.predict(X_log.iloc[valid_idx])

xgb_mean_oof = np.mean([xgb_oofs[seed] for seed in selected_seeds], axis=0)

validation_summary = pd.DataFrame(
    [
        {"方案": "lasso", "OOF_RMSLE": rmsle(y, lasso_oof)},
        {"方案": "xgb_seed_23", "OOF_RMSLE": rmsle(y, xgb_oofs[23])},
        {"方案": "xgb_seed_73", "OOF_RMSLE": rmsle(y, xgb_oofs[73])},
        {"方案": "xgb_seed_9", "OOF_RMSLE": rmsle(y, xgb_oofs[9])},
        {"方案": "xgb_seed_mean", "OOF_RMSLE": rmsle(y, xgb_mean_oof)},
        {"方案": "0.60_lasso_0.40_xgb_mean", "OOF_RMSLE": rmsle(y, 0.60 * lasso_oof + 0.40 * xgb_mean_oof)},
    ]
)
validation_summary

,方案,OOF_RMSLE
0,lasso,0.110142
1,xgb_seed_23,0.114403
2,xgb_seed_73,0.114218
3,xgb_seed_9,0.114131
4,xgb_seed_mean,0.113787
5,0.60_lasso_0.40_xgb_mean,0.107416


## 4. 生成提交文件

In [6]:
final_lasso = clone(lasso_model)
final_lasso.fit(X_log, y)

xgb_test_predictions = []
for seed in selected_seeds:
    final_xgb = build_model(
        XGBRegressor(random_state=seed, **xgb_params),
        scale_numeric=False,
    )
    final_xgb.fit(X_log, y)
    xgb_test_predictions.append(final_xgb.predict(X_test_log))

xgb_mean_prediction = np.mean(xgb_test_predictions, axis=0)
final_prediction = np.maximum(0.60 * final_lasso.predict(X_test_log) + 0.40 * xgb_mean_prediction, 0)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": final_prediction})
SUBMISSION_DIR.mkdir(exist_ok=True)
submission_path = SUBMISSION_DIR / "10_submission_xgboost_seed_ensemble.csv"
submission.to_csv(submission_path, index=False)

submission_path, submission.head(), submission["SalePrice"].describe()

(WindowsPath('C:/Users/84740/house-prices-advanced-regression/submissions/10_submission_xgboost_seed_ensemble.csv'),
      Id      SalePrice
 0  1461  120806.086296
 1  1462  154308.872886
 2  1463  183383.630540
 3  1464  197413.027542
 4  1465  191223.616925,
 count      1459.000000
 mean     178680.119686
 std       78718.076980
 min       42240.738552
 25%      126462.426434
 50%      157505.270920
 75%      210605.031984
 max      854564.682238
 Name: SalePrice, dtype: float64)

## 5. 提交命令

```powershell
kaggle competitions submit -c house-prices-advanced-regression-techniques -f submissions/10_submission_xgboost_seed_ensemble.csv -m "xgboost seed ensemble lasso blend"
```